# 86 — FNSPID assigned-window outcome-leg decomposition

**Objective.** Test the external-review timing alternative: does the development negative-share association occur before the assigned open, during the assigned session, or after the assigned close?

This is a post-review diagnostic specified after the headline results and after the evaluation block was opened. It is not independent confirmation, cannot recover row-level story-arrival times, and does not execute Notebook 75, rescore text, rebuild the checkpoint, or pool FNSPID with LSEG.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("run the notebook from inside the repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.lib.outcome_leg_decomposition import run  # noqa: E402

SPEC_PATH = ROOT / "experiments/specs/fnspid_outcome_leg_decomposition_v1_20260816.json"
spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
assert spec["status"] == "frozen_before_outcome_leg_results_after_external_review"
spec["frozen_at_utc"], spec["git_commit_at_freeze"]


('2026-08-16T13:43:25Z', 'acaf530c75a20e882166e7218245342384cf2b1b')

## Frozen plan

- Reproduce the assigned-open-to-next-open baseline from adjusted open and close prices.
- Decompose it into assigned-session intraday (open to close) and post-close overnight (close to next open) legs; estimate the pre-open overnight leg separately.
- Use the same daily cross-sectional centred-rank model, mean-sentiment and story-count controls, minimum 10 names, and HAC(5) inference.
- Apply Benjamini–Hochberg at q=0.05 across exactly the three new development legs. Evaluation-leg and era-contrast families are post-opening diagnostics.
- Reconcile the tied-rank effect translation, realised evaluation precision, singleton stratum, and omitted FNSPID drawdown from frozen aggregate evidence.
- Preserve nulls and adverse results. The interpretation gate in the specification controls the manuscript wording.


In [2]:
written = run()
tables = {
    name: pd.read_csv(path)
    for name, path in written.items()
    if path.suffix == ".csv"
}
manifest = json.loads(written["manifest"].read_text(encoding="utf-8"))
display(Markdown("## Coverage and arithmetic identity"))
display(tables["coverage_audit"])
display(tables["timing_identity_audit"])


## Coverage and arithmetic identity

,item,value
0,panel_rows,7.155460e+05
1,panel_symbols,5.700000e+02
2,price_rows,1.950672e+06
3,missing_price_symbols,0.000000e+00
4,development_rows,5.121530e+05
5,development_sessions,2.264000e+03
6,development_singleton_share,4.831955e-01
7,development_assigned_open_to_next_open_complet...,5.121490e+05
8,development_assigned_session_intraday_complete...,5.121530e+05
9,development_post_close_overnight_complete_rows,5.121490e+05


,check,n_complete,max_absolute_error,tolerance,passes
0,stock_gross_return_identity,1946568,1.776357e-15,1.000000e-12,True
1,market_gross_return_identity,3520,2.220446e-16,1.000000e-12,True
2,recomputed_abnormal_full_window_matches_commit...,715534,0.000000e+00,1.000000e-12,True


## Timing results

The baseline row is an audit reproduction. The three leg rows are the declared post-review family. Evaluation and era contrasts remain explicitly post-opening and exploratory.


In [3]:
coefficient_columns = [
    "regime",
    "outcome_leg",
    "estimate",
    "se",
    "ci_low",
    "ci_high",
    "p_two_sided",
    "q_bh_three_leg",
    "bh_reject_q05",
    "n_clusters",
]
display(tables["coefficient_summary"][coefficient_columns])
display(Markdown("### Direct era contrasts"))
display(
    tables["era_contrasts"][
        [
            "outcome_leg",
            "reference_estimate",
            "comparison_estimate",
            "estimate",
            "ci_low",
            "ci_high",
            "p_two_sided",
            "q_bh_three_leg",
        ]
    ]
)


,regime,outcome_leg,estimate,se,ci_low,ci_high,p_two_sided,q_bh_three_leg,bh_reject_q05,n_clusters
0,development,assigned_open_to_next_open,-0.008310,0.002954,-0.014100,-0.002520,0.004907,NaN,False,2264
1,development,assigned_session_intraday,-0.010291,0.002892,-0.015960,-0.004622,0.000374,0.001121,True,2264
2,development,post_close_overnight,0.001548,0.002911,-0.004158,0.007254,0.594922,0.594922,False,2264
3,development,pre_open_overnight,-0.006381,0.003194,-0.012640,-0.000121,0.045719,0.068578,False,2264
4,evaluation,assigned_open_to_next_open,0.005833,0.005150,-0.004261,0.015928,0.257393,NaN,False,998
5,evaluation,assigned_session_intraday,0.003373,0.005132,-0.006687,0.013432,0.511114,0.511114,False,998
6,evaluation,post_close_overnight,0.003478,0.004919,-0.006164,0.013120,0.479586,0.511114,False,998
7,evaluation,pre_open_overnight,-0.015517,0.004945,-0.025210,-0.005825,0.001702,0.005107,True,998


### Direct era contrasts

,outcome_leg,reference_estimate,comparison_estimate,estimate,ci_low,ci_high,p_two_sided,q_bh_three_leg
0,assigned_open_to_next_open,-0.008310,0.005833,0.014143,0.002499,0.025788,0.017284,NaN
1,assigned_session_intraday,-0.010291,0.003373,0.013664,0.002113,0.025214,0.020423,0.061269
2,post_close_overnight,0.001548,0.003478,0.001930,-0.009275,0.013135,0.735700,0.735700
3,pre_open_overnight,-0.006381,-0.015517,-0.009136,-0.020668,0.002396,0.120467,0.180701


## Required corrections

These tables replace the unattainable 0.8 rank-change translation, distinguish prospective from realised precision, isolate singleton firm-days, and report the adverse overall FNSPID drawdown comparison.


In [4]:
for heading, key in (
    ("Rank-effect translation", "rank_effect_translation"),
    ("Power reconciliation", "power_reconciliation"),
    ("Story-count strata", "story_count_strata"),
    ("FNSPID drawdown reconciliation", "drawdown_reconciliation"),
):
    display(Markdown(f"### {heading}"))
    display(tables[key])


### Rank-effect translation

,translation,rank_change,coefficient,fitted_rank_change,fitted_percentile_points,valid_for_observed_tied_regressor,note
0,hypothetical_fixed_0_8,0.800000,-0.00914,-0.007312,-0.731231,False,superseded hypothetical translation; exceeds t...
1,pooled_observed_p90_minus_p10,0.560560,-0.00914,-0.005124,-0.512374,True,primary corrected translation on exact Noteboo...
2,mean_session_specific_p90_minus_p10,0.527215,-0.00914,-0.004819,-0.481895,True,sensitivity averaging each session's observed ...


### Power reconciliation

,role,method,two_sided_alpha,effect,standard_error,effect_z,power,mde80,se_ratio_to_projection
0,prospective_before_evaluation,development HAC SE scaled by square-root sessi...,0.025,-0.00831,0.004449,1.867700,0.354332,0.013717,1.000000
1,realised_precision_reconciliation,observed evaluation HAC(5) standard error; pos...,0.025,-0.00831,0.005150,1.613448,0.265075,0.015879,1.157583


### Story-count strata

,stratum,firm_days,story_count_control_included,coefficient,n_clusters,hac_lags,inference,estimate,se,t,...,n_rows_complete,minimum_names,expected_direction,direction_matches,dates_total,dates_used,dates_below_min_names,dates_constant_variable,dates_rank_deficient,comparability_note
0,n = 1,247470,False,beta_negative_share,2263,5,mean_daily_cross_sectional_rank_coefficient_hac,-0.010252,0.004684,-2.188679,...,247470,10,negative,True,2264,2263,0,1,0,descriptive only; ranks are recomputed within ...
1,n >= 2,264683,True,beta_negative_share,2264,5,mean_daily_cross_sectional_rank_coefficient_hac,-0.005291,0.003792,-1.395402,...,264679,10,negative,True,2264,2264,0,0,0,descriptive only; ranks are recomputed within ...
2,n >= 3,137142,True,beta_negative_share,2262,5,mean_daily_cross_sectional_rank_coefficient_hac,-0.001828,0.006134,-0.298096,...,137140,10,negative,True,2264,2262,1,1,0,descriptive only; ranks are recomputed within ...


### FNSPID drawdown reconciliation

,period,control_arm,overlay_arm,control_max_drawdown,overlay_max_drawdown,drawdown_relative_improvement,overlay_has_smaller_drawdown
0,2020_2023,control_har_target,har_sentiment,-0.157300,-0.163625,-0.040212,False
1,2020,control_har_target,har_sentiment,-0.129468,-0.073924,0.429013,True
2,2021_2023,control_har_target,har_sentiment,-0.157300,-0.163625,-0.040212,False


## Interpretation gate and next step

The machine-readable gate below selects one of the four wording paths frozen before execution. It cannot identify the true story-arrival mechanism because the checkpoint lacks reliable row-level availability timestamps.


In [5]:
gate = manifest["interpretation_gate"]
display(Markdown(f"**Frozen gate decision:** `{gate['decision']}`"))
display(pd.DataFrame([gate]))
display(
    Markdown(
        "Carry this result into the specification, aggregate result snapshot, evidence map, "
        "manuscript, and revision record together. Do not upgrade it to causal timing, "
        "independent confirmation, or a trading claim."
    )
)


**Frozen gate decision:** `intraday_only_or_dominant`

,decision,assigned_session_intraday_interval_negative,post_close_overnight_interval_negative,pre_open_overnight_interval_negative,assigned_session_intraday_bh_reject,post_close_overnight_bh_reject,pre_open_overnight_bh_reject
0,intraday_only_or_dominant,True,False,True,True,False,False


Carry this result into the specification, aggregate result snapshot, evidence map, manuscript, and revision record together. Do not upgrade it to causal timing, independent confirmation, or a trading claim.